# Find large files

In [ ]:
#@title Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Find
folder = "/content/drive/MyDrive" #@param {type:"string"}
top = 20 #@param {type:"integer"}

# Accept paths as Drive shows them (Temp\video.mp4) or as full /content paths.
folder = folder.strip().strip('"').replace("\\", "/")
if folder and not folder.startswith("/"):
    folder = "/content/drive/MyDrive/" + folder.removeprefix("MyDrive/").removeprefix("My Drive/")

import os, heapq
def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024

root = folder.rstrip("/")
if not os.path.isdir(root):
    raise SystemExit(f"Not a folder: {root}")
files, dirs = [], {}
for dirpath, _, names in os.walk(root):
    for name in names:
        p = os.path.join(dirpath, name)
        try:
            size = os.path.getsize(p)
        except OSError:
            continue
        files.append((size, p))
        d = dirpath
        while True:
            dirs[d] = dirs.get(d, 0) + size
            if d == root:
                break
            d = os.path.dirname(d)
print("Total:", human(dirs.get(root, 0)))
print("\nFiles:")
for size, p in heapq.nlargest(top, files):
    print(f"{human(size):>10}  {os.path.relpath(p, root)}")
print("\nFolders:")
for d, size in heapq.nlargest(top, ((d, s) for d, s in dirs.items() if d != root), key=lambda x: x[1]):
    print(f"{human(size):>10}  {os.path.relpath(d, root)}/")
